# Axis-aligned and oblique forest comparison

Fit each parent once on a deterministic rotated-signal simulation and compare the same declared original relation.


All examples use local synthetic data, fixed seeds, and a reduced smoke
configuration by default. `original`, `kerf`, `path`, `random_height`, and the
separately named isolation `collision` relation are symmetric affinities.
`rfgap`, `honest_gap`, and conditional weights are directed query-to-training
weights; downstream visualization, clustering, or attribution must state any
symmetrization. Prediction identity is claimed only for unsymmetrized `rfgap`
or `honest_gap` under their declared bootstrap/honesty contracts.

In [1]:
SMOKE = True
RANDOM_STATE = 0
N_SAMPLES = 80 if SMOKE else 800
N_ESTIMATORS = 20 if SMOKE else 300

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=N_SAMPLES, n_features=8, n_informative=5,
    n_redundant=1, random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
)

In [3]:
from sklearn.ensemble import RandomForestClassifier
from forestgap import Proximity

families = [("axis", RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE))]
try:
    from treeple import ObliqueRandomForestClassifier
    families.append(("oblique", ObliqueRandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=1)))
except ImportError:
    pass

results = []
for name, forest in families:
    proximity = Proximity(forest, relation="original").fit(X_train, y_train)
    results.append({"family": name, "accuracy": float((forest.predict(X_test) == y_test).mean()), "density": proximity.diagnostics()["density"]})
results

[{'family': 'axis', 'accuracy': 0.8, 'density': 0.7122222222222222},
 {'family': 'oblique', 'accuracy': 0.75, 'density': 0.7627777777777778}]